# 01 — From a Logistics Graph to a QUBO

Every problem in this hackathon — vehicle routing, warehouse allocation, whatever comes next — is mapped to the same canonical form:

$$E(x) = \underbrace{x^T Q_{obj}\, x}_{\text{true cost}} + \sum_c \lambda_c \underbrace{x^T Q_c\, x}_{\text{penalty for constraint } c}$$

The toolkit keeps these pieces **separate** on purpose: the $\lambda_c$ weights are *your* knob (notebook 04 tunes them automatically).

In [ ]:
import sys; sys.path.insert(0, "../..")
from problems.vrp import VehicleRoutingProblem

problem = VehicleRoutingProblem.small_instance()
print("customers:", problem.num_customers, "| vehicles:", problem.num_vehicles)
print("binary variables:", problem.num_vars(), "  (x[c,v] = 1 iff customer c rides on vehicle v)")
print("constraints:", [c.name for c in problem.constraints()])

In [ ]:
# The pure objective — NO penalties baked in.
objective = problem.objective_qubo()
print("objective terms:", len(objective.terms))

# Penalties enter only when YOU choose lambdas:
qubo_soft = problem.build_qubo({"one_hot": 1.0, "capacity": 1.0})
qubo_hard = problem.build_qubo({"one_hot": 50.0, "capacity": 50.0})

# Same infeasible assignment (customer 0 unassigned), very different energy:
bits = [0] * problem.num_vars()
for c in range(1, 3): bits[problem.var(c, 0)] = 1
for c in range(3, 6): bits[problem.var(c, 1)] = 1
print("energy at lambda=1: ", qubo_soft.energy(bits))
print("energy at lambda=50:", qubo_hard.energy(bits))

## The lambda dilemma (this is the whole game)

- $\lambda$ **too small** → the minimum of $E(x)$ is an *illegal* solution: the QPU happily drops customers to save distance.
- $\lambda$ **too large** → penalties dwarf the objective; the energy landscape flattens and the QPU can no longer tell a cheap legal solution from an expensive one.

⚠️ **Known wart, on purpose:** the shipped `CapacityConstraint` uses an *equality-style* penalty $\lambda(\text{load}-\text{cap})^2$, which also punishes **under**loaded vehicles. Encoding the true inequality needs slack variables — that is a documented stretch goal worth real points. See `problems/common.py`.